# Introduction
The main purpose of this notebook is to identify the most suitable companies for **sentiment analysis** using [historical news dataset](https://huggingface.co/datasets/Zihan1004/FNSPID). We focus on companies that have **long news coverage** and a **large number of valid (non-empty) articles**.
The selection criteria are:

### 1. Coverage period of news
Prefer companies with **long historical coverage**, ideally **more than 10 years**, and covering **earlier periods** (e.g., 2000–2010 or 2010–2020).

### 2. Volume of valid news entries
Prefer companies with a **high number of usable articles**, ensuring enough **data for sentiment analysis**.

# Methodology

### 1. **Large dataset processing**
   
The financial news dataset is very large (**23.2 GB**). Because of this, it **cannot be fully loaded into a DataFrame** without causing **memory issues**. To avoid this, the dataset is **processed in smaller chunks**.

### 2. **Measure useful news per company**
 
To select a company for sentiment analysis, we need to ensure that it has **good news coverage over a long period of time** and a **large number of useful (non-empty) articles**. For this reason, the main purpose of processing the dataset in chunks is to measure, for each company, the **total number of news entries**, the **number of non-empty articles**, and the **length of available news coverage in years** (from the first to the last article).


### 3. **Rank companies for sentiment analysis**
   
After we gather the information for each company in the dataset, we calculate the **quality** of the data and an overall **score** for sentiment analysis.


The **quality** is calculated as the **ratio of non-empty news articles** to the **total number of news entries** for a company. This shows how much of the data is actually **usable** for sentiment analysis.
The **score** is calculated by combining three factors:

- **Length of news coverage** - how many years the company has news available
- **Number of non-empty articles** - the total count of usable news articles
- **Data quality** - the proportion of valid articles
      
The **coverage length** and **article count** are first **normalized** so that companies can be compared fairly. These values are then **weighted and summed** to produce a **final score**, which is used to **rank the companies**.

After calculating the **quality** and **score** for each company, we display the **top 20 companies** with the highest score and **select 2 companies** for creating the sentiment analysis dataset.

### 4. **Export selected company data**
After the **two companies** are chosen, we **save their data** into separate datasets, creating **one dataset for each company**. This allows us to use their news data directly for **sentiment analysis**.

### Import Libraries

In [1]:
import pandas as pd
from collections import defaultdict
from pathlib import Path

### Function to process a chunk of data

This function processes a part of the dataset and calculates basic statistics for each company:

1. Finds the **earliest and latest news dates**
2. Counts the **number of non-empty articles**
3. Counts the **total number of articles**

In [2]:
def process_chunk_of_dataset(chunk):
    # Make a copy right away to avoid SettingWithCopyWarning later
    chunk_copy = chunk.copy()
    
    # Convert the 'Date' column to datetime64 format
    chunk_copy["Date"] = pd.to_datetime(chunk_copy["Date"], errors="coerce", utc=True)
    
    # Remove rows with missing a values in 'Stock_symbol' or 'Date' columns
    chunk_copy = chunk_copy.dropna(subset=["Stock_symbol", "Date"])

    # Find strings in 'Aticle' column that are not empty or NaN
    article = chunk_copy["Article"].astype(str)
    #chunk["NonEmpty"] = article.str.strip().ne("") & article.ne("nan")
    chunk_copy.loc[:, "NonEmpty"] = article.str.strip().ne("") & article.ne("nan")

    # Create summary for each company in the current chunk
    processed_summary = chunk_copy.groupby("Stock_symbol").agg(
        min_date=("Date", "min"),
        max_date=("Date", "max"),
        Articles=("NonEmpty", "sum"),
        Total_rows=("NonEmpty", "count")
    )

    return processed_summary

### Function to update a company statistics

This function takes the statistics from the current data chunk and **updates the overall stats for each company**:
1. Updates the **earliest and latest news dates** across all chunks
2. Adds the **number of non-empty articles** to the total
3. Adds the **total number of articles** to the total

In [3]:
def update_global_stats(global_stats, processed_summary):
    # Iterate over processed_summary dictionary
    for company, row in processed_summary.iterrows():
        # Get a reference to the current company stats from global stats
        company_stats = global_stats[company]

        # Update min/max dates for the current company in the global stats via reference
        if company_stats["min"] is None or row["min_date"] < company_stats["min"]:
            company_stats["min"] = row["min_date"]
            
        if company_stats["max"] is None or row["max_date"] > company_stats["max"]:
            company_stats["max"] = row["max_date"]

        # Update article counts for the current company in the global stats via reference
        company_stats["Articles"] += row["Articles"]
        company_stats["Total_rows"] += row["Total_rows"]

### Function to convert collected statistics into a DataFrame

This function takes the collected statistics for all companies and **creates a summary table**:
1. Calculates **news coverage length in years** for each company
2. Includes the **total number of non-empty articles** and total articles**
3. Computes the **data quality** as the ratio of non-empty articles to total articles
4. Returns a **DataFrame** with all these metrics for further analysis

In [4]:
def convert_global_stats_to_dataframe(global_stats):
    # Create list of rows
    rows = []

    # Iterate over items in global stats
    for company, company_stats in global_stats.items():
        # Skip companies with no news data
        if company_stats["min"] is None:
            continue

        # Calculate total number of years current company has news for
        years = (company_stats["max"] - company_stats["min"]).days / 365.25

        # Append the current companies stats into the rows list
        rows.append({
            "Stock_symbol": company,
            "Start": company_stats["min"].to_pydatetime(),
            "End": company_stats["max"].to_pydatetime(),
            "Years": years,
            "Articles": company_stats["Articles"],
            "Total_rows": company_stats["Total_rows"],
            "Quality": company_stats["Articles"] / company_stats["Total_rows"] if company_stats["Total_rows"] else 0
        })

    # Return total stats as DataFrame
    return pd.DataFrame(rows)

### Function to compute company statistics.

This is the main function that **processes the entire dataset efficiently**:
1. Reads the CSV file **in smaller chunks** to avoid memory issues
2. Processes each chunk to get **per-company statistics**
3. **Updates global stats** with results from each chunk
4. Converts the final statistics into a **DataFrame** for all companies

In [5]:
def create_global_stats(input_csv, chunk_size=100_000):
    # Create a dictionary to store stats for all companies from the dataset
    # Each company will have the following data: min date, max date, number of valid articles, total rows
    global_stats = defaultdict(lambda: {
        "min": None,
        "max": None,
        "Articles": 0,
        "Total_rows": 0
    })

    # Read only data from columns that are related to creation of stats
    usecols = ["Stock_symbol", "Date", "Article"]

    # Read the dataset in chunks to ensure we avoid issues with memory as the dataset is too big
    for chunk in pd.read_csv(input_csv, chunksize=chunk_size, usecols=usecols):
        # Process the current chunk
        processed_summary = process_chunk_of_dataset(chunk)
        
        # Update the global stats with data from the current chunk
        update_global_stats(global_stats, processed_summary)

    # Convert global stats dictionary into a DataFrame
    df = convert_global_stats_to_dataframe(global_stats)

    return df


### Function to calculate normalized scores and rank companies

This function calculates a **combined score** for each company to help with ranking:
1. **Normalizes the news coverage length and number of articles so companies can be compared fairly
2. Combines **coverage**, **article count**, and **data quality** into a **weighted score**
3. **Sorts companies** from highest to lowest score for ranking

In [6]:
def add_scores(df):
    # Normalize the 'Years' and 'Articles' columns to have values between 0 and 1
    df["Years_norm"] = df["Years"] / df["Years"].max()
    df["Articles_norm"] = df["Articles"] / df["Articles"].max()
    
    # Calculate a score for all the companies using weighted values 
    # Current weiths are:
    # Years = 40% 
    # Valid Articles = 40%
    # Quality = 20%
    # With the current weights we are saying that Years and number of valid articles are more important than the quality.
    df["Score"] = 0.4 * df["Years_norm"] + 0.4 * df["Articles_norm"] + 0.2 * df["Quality"]
    
    # Sort companies by score in descending order
    df_ranked = df.sort_values("Score", ascending=False).reset_index(drop=True)
    
    return df_ranked

### Set path to the dataset, create chunk size and call the orchestrator.

This section runs the main pipeline and displays the results:
1. Sets the **input CSV file** and **chunk size** for processing
2. Computes **company statistics** from the dataset using chunks
3. Calculates **normalized scores** and **ranks companies**
4. Displays the **top 25 companies** based on their score, showing: **Stock symbol**, **years of coverage**, **number of articles**, **data quality**, **and final score**

In [7]:
# path to the dataset. Currently the dataset is located in the same folder as the notebook
input_csv = Path("./Inputs/nasdaq_exteral_data.csv")
# The size of the chunks to process
chunk_size = 10_000

# Create global stats and populate it
df_global_stats = create_global_stats(input_csv, chunk_size=chunk_size)

# Add scores to the the global stats
df_ranked = add_scores(df_global_stats)

# Show top 25 companies
df_ranked.head(25)[["Stock_symbol", "Years", "Articles", "Quality", "Score"]]

,Stock_symbol,Years,Articles,Quality,Score
0,BRK,14.157426,8797,1.000000,0.659251
1,CLSN,14.214921,8472,0.930376,0.633392
2,GE,11.655031,8680,1.000000,0.633085
3,T,7.488022,9449,0.998521,0.626087
4,GS,7.890486,8730,1.000000,0.602366
5,BROGW,0.043806,10456,1.000000,0.600380
6,AMD,7.074606,8959,0.972853,0.598625
7,CVX,7.367556,8688,1.000000,0.596227
8,NKE,14.064339,7129,1.000000,0.594634
9,NVDA,12.788501,8716,0.734783,0.591243


From the ranked list of companies, we selected **BRK** and **NVDA** for sentiment analysis:
- **BRK (Berkshire Hathaway)** is a large **investment company** with **14.16 years of news coverage**, **8,797 valid articles**, and a **quality of 1.0**, which made it the top-ranked company among all candidates.
- **NVDA (NVIDIA Corporation)** is a leading **technology company** specializing in **chip development**. It has **12.79 years of news coverage**, **8,716 valid articles**, and a **quality of 0.73**, reflecting a slightly lower proportion of valid news and placing it ninth among the candidates. Despite this, it is interesting to include a **major tech company** to compare with the **financial sector**.

By selecting these two companies, we cover **both finance and technology sectors**, providing a **diverse and meaningful dataset** for sentiment analysis.


---

# Save company to CSV file

In [8]:
def save_company_from_csv(input_csv, stock_symbol, output_path, chunk_size=100_000):
    output_path = Path(output_path)

    if output_path.exists():
        output_path.unlink()

    header_written = False

    for chunk in pd.read_csv(input_csv, chunksize=chunk_size):
        company_chunk = chunk[chunk["Stock_symbol"] == stock_symbol]

        if not company_chunk.empty:
            company_chunk.to_csv(
                output_path,
                mode="a",
                index=False,
                header=not header_written
            )
            header_written = True

In [9]:
# Save NVDA's data
save_company_from_csv(input_csv, "NVDA", Path("./Outputs/NVDA_news.csv"), chunk_size)

# Save NVDA's data
save_company_from_csv(input_csv, "BRK", Path("./Outputs/BRK_news.csv"), chunk_size)